In [ ]:
## Problem 4: Basic LP
#### (b) Plot the feasible region

In [ ]:
using Plots

# Define the constraints as functions of y1
f1(y1) = (2 - 4*y1)
f2(y1) = (1 - y1)/2
f3(y1) = (4 - 2*y1)
f4(y1) = (15 - 3*y1)/5

# Plot the constraints
y1_vals = 0:0.01:10  # Define the range for y1
plot(y1_vals, f1.(y1_vals), label="4y1 + y2 = 2", linewidth=2, color=:red)
plot!(y1_vals, f2.(y1_vals), label="y1 + 2y2 = 1", linewidth=2, color=:blue)
plot!(y1_vals, f3.(y1_vals), label="2y1 + y2 = 4", linewidth=2, color=:green)
plot!(y1_vals, f4.(y1_vals), label="3y1 + 5y2 = 15", linewidth=2, color=:purple)

# Add axes labels and title
xlabel!("y1")
ylabel!("y2")
title!("Feasible Region for the Constraints")

# Add limits to the axes
xlims!(0, 5)
ylims!(0, 5)

# Show the plot
plot!()


## Problem 5
#### (b) Solve the Linear system

In [42]:
using JuMP, Gurobi

branches = 0:9
availability = [1000000, 90000, 20000, -5000, -450000, -67000, -190000, 6700, -10000, -140000]

# Distances matrix and number of branches (including vault)
distances = [
    0 29 83 60 40 39 33 43 84 21;
    29 0 56 95 62 95 76 26 67 28;
    83 56 0 36 69 50 44 44 26 28;
    60 95 36 0 63 82 34 69 78 40;
    40 62 69 63 0 24 56 28 85 37;
    39 95 50 82 24 0 38 63 73 66;
    33 76 44 34 56 38 0 53 49 25;
    43 26 44 69 28 63 53 0 41 19;
    84 67 26 78 85 73 49 41 0 43;
    21 28 28 40 37 66 25 19 43 0
]

transport_costs = 0.001 * distances

model = Model(Gurobi.Optimizer)

# Decision variables: Money transferred from i to j
@variable(model, x[branches, branches] >= 0)

# Objective: Minimize transportation cost
@objective(model, Min, sum(transport_costs[i+1,j+1] * x[i,j] for i in branches, j in branches))

for i in 1:9  # 0 is the vault
    @constraint(model, sum(x[j,i] for j in branches) - sum(x[i,j] for j in branches) + availability[i+1] == 0)
end

# Constraint: Flow limit between any two branches/vault
for i in branches, j in branches
    @constraint(model, x[i, j] <= 100000)
end

# Solve the model
optimize!(model)

# (i)
# Print the results
println("Objective value (minimum transportation cost): ", objective_value(model))
println("Optimal Cash Transfers:")
for i in branches, j in branches
    if value(x[i, j]) > 0
        println("Transfer from Branch $i to Branch $j (in thousands): ", 1/1000 * value(x[i, j]))
    end
end

# Calculate final cash in vault after all transactions
initial_cash_vault = 1000000
cash_sent_from_vault = sum(value(x[0, j]) for j in 1:9) # Cash sent from vault to branc
cash_received_by_vault = sum(value(x[i, 0]) for i in 1:9) # Cash received by vault from
final_cash_in_vault = initial_cash_vault - cash_sent_from_vault + cash_received_by_vault
println("Final amount of cash in the vault: \$$(final_cash_in_vault)") 

Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-19
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (mac64[x86] - Darwin 24.0.0 24A348)

CPU model: Intel(R) Core(TM) i5-8279U CPU @ 2.40GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 109 rows, 100 columns and 262 nonzeros
Model fingerprint: 0x3eb8e38b
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 1e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+03, 4e+05]
Presolve removed 100 rows and 10 columns
Presolve time: 0.00s
Presolved: 9 rows, 90 columns, 162 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.223375e+05   0.000000e+00      0s
      21    5.1824600e+04   0.000000e+00   0.000000e+00      0s

Solved in 21 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.182460000e+04

User-callback calls 65, time in user-callback 0.

## (c)

In [49]:
# Update specific entries to the discounted rate for Branch 1 and Branch 3
new_transport_costs = transport_costs
new_transport_costs[2, 4] = 0.0008 * distances[2, 4]  # From Branch 1 to Branch 3
new_transport_costs[4, 2] = 0.0008 * distances[4, 2]  # From Branch 3 to Branch 1

# Create the optimization model
model_c = Model(Gurobi.Optimizer)

# Define variables: x[i, j] represents the amount of cash transferred from branch i to branch j
@variable(model_c, x[branches, branches] >= 0)

# Objective: Minimize the total transportation cost
@objective(model_c, Min, sum(new_transport_costs[i+1, j+1] * x[i, j] for i in branches, j in branches))

# Constraints for each branch (i = 1 to 9):
# Total cash arriving at a branch minus total cash leaving + availability = 0
for i in 1:9
    @constraint(model_c, sum(x[j, i] for j in branches) - sum(x[i, j] for j in branches) + availability[i+1] == 0)
end

# Vault does not need to be balanced, no constraint needed for it

# Each transaction can carry at most $100,000
for i in branches, j in branches
    @constraint(model_c, x[i, j] <= 100000)
end

# Solve the model
optimize!(model_c)

# Print the results
println("Objective value (minimum transportation cost with discount): ", objective_value(model1))
println("Optimal Cash Transfers (with discounted rates):")
for i in branches, j in branches
    if value(x[i, j]) > 0
        println("Transfer from Branch $i to Branch $j: ", value(x[i, j]))
    end
end


Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-19
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (mac64[x86] - Darwin 24.0.0 24A348)

CPU model: Intel(R) Core(TM) i5-8279U CPU @ 2.40GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 109 rows, 100 columns and 262 nonzeros
Model fingerprint: 0xadd5ed8e
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 1e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+03, 4e+05]
Presolve removed 100 rows and 10 columns
Presolve time: 0.00s
Presolved: 9 rows, 90 columns, 162 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.223375e+05   0.000000e+00      0s
      21    5.1824600e+04   0.000000e+00   0.000000e+00      0s

Solved in 21 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.182460000e+04

User-callback calls 65, time in user-callback 0.

#### (d)

In [75]:
# Create the optimization model
model2 = Model(Gurobi.Optimizer)

# Define variables: x[i, j] represents the amount of cash transferred from branch i to branch j
@variable(model2, x[branches, branches] >= 0)

# Define a binary variable for the loan decision (1 = take loan, 0 = no loan)
@variable(model2, L, Bin)

# Objective: Minimize transportation cost or Boundz cost + flat fee for loan
@objective(model2, Min, sum(transport_costs[i+1, j+1] * x[i, j] for i in branches, j in branches) + 50000 * L)

# Vault constraint: Total cash sent from the vault should not exceed its supply
@constraint(model2, sum(x[0, j] for j in 1:9) <= 1_000_000)
    
# Balance constraints for each branch (including additional cash flow if the loan is taken)
for i in 1:9
    if i == 4
        @constraint(model2, sum(x[j, i] for j in branches) - sum(x[i, j] for j in branches) + availability[i+1] + 5000 * L == 0)
    else
        @constraint(model2, sum(x[j, i] for j in branches) - sum(x[i, j] for j in branches) + availability[i+1] == 0)
    end
end

# Each transaction can carry at most $100,000
for i in branches, j in branches
    @constraint(model2, x[i, j] <= 100000)
end

# Solve the model
optimize!(model2)



Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-19
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (mac64[x86] - Darwin 24.0.0 24A348)

CPU model: Intel(R) Core(TM) i5-8279U CPU @ 2.40GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 110 rows, 101 columns and 272 nonzeros
Model fingerprint: 0x09f1b8bd
Variable types: 100 continuous, 1 integer (1 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+03]
  Objective range  [2e-02, 5e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+03, 1e+06]
Presolve removed 101 rows and 10 columns
Presolve time: 0.00s
Presolved: 9 rows, 91 columns, 163 nonzeros
Variable types: 90 continuous, 1 integer (1 binary)
Found heuristic solution: objective 51824.600000

Root relaxation: cutoff, 25 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | I